# Pecca quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/quickstart.ipynb)

Replace a repetitive LLM call with a small, auditable model: **connect → profile → train → shadow → status → audit pack**.
Everything runs offline-friendly on a synthetic dataset (a fictional "Northbridge Bank"); the "LLM" is a stand-in function.

In [1]:
%pip install -q pecca huggingface_hub

In [2]:
import os, tempfile

work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"   # no progress bars in the notebook output
os.environ.setdefault("PECCA_TEST_SMALL", "0")  # real multilingual-e5 embeddings (set to 1 for a tiny offline stand-in)

import pandas as pd
import pecca
print("pecca", pecca.__version__)

pecca 0.1.0


## 1. Get the demo data
The dataset lives on the Hugging Face Hub; if you are offline we generate the same data locally.

In [3]:
try:
    from huggingface_hub import hf_hub_download
    path = hf_hub_download("pecca-core/demo-support-emails", "train.parquet", repo_type="dataset")
    source = "Hugging Face Hub"
except Exception:  # offline / no hub access
    from pecca.data.synthetic import generate
    path = os.path.join(work, "train.parquet")
    generate(6000).to_parquet(path)
    source = "generated locally"
print("data:", source)

data: Hugging Face Hub


## 2. Connect
Map your columns to Pecca's canonical schema: `input`, `llm_output`, `human_label`, `call_name`.

In [4]:
ds = pecca.connect(
    "parquet",
    path=path,
    columns={
        "input": {"subject": "email_subject", "body": "email_body"},
        "llm_output": "llm_rfi",          # what the LLM answered
        "human_label": "human_rfi",       # agent corrections (present for ~55% of rows)
        "call_name": {"literal": "route_rfi"},
    },
    input_template="{subject}\n\n{body}",
)
print(ds)
ds.sample(3)[["input", "llm_output", "human_label"]]

Dataset(rows=6000, calls=['route_rfi'])


,input,llm_output,human_label
0,{'subject': 'My payment sent on 27 jul never r...,transfer_not_received,NaN
1,{'subject': 'Re: How do i activate my new card...,card_replacement,card_activation
2,{'subject': 'I want a recurring payment every ...,standing_order_setup,NaN


## 3. Profile
Deterministic rules decide whether the call can be replaced.

In [5]:
profile = pecca.profile(ds).profiles["route_rfi"]
{k: profile.to_dict()[k] for k in ("task_type", "input_type", "n_rows", "n_unique_outputs", "replaceable", "languages", "min_class_count")}

{'task_type': 'classification',
 'input_type': 'multi_text',
 'n_rows': 6000,
 'n_unique_outputs': 60,
 'replaceable': True,
 'languages': ['en', 'tl', 'es'],
 'min_class_count': 81}

## 4. Train
A tournament over the small candidates on identical cross-validated splits (the slow fine-tuned `xlmr_finetune` is left out for a laptop/Colab CPU).

In [6]:
result = pecca.train("route_rfi", ds, candidates=["tfidf_linear", "e5_logreg"])
print(f"winner {result.winner}: {result.metric_name} {result.metric:.3f} (LLM vs human labels: {result.llm_metric:.3f})")
print(f"threshold {result.threshold:.2f}, expected fallback to the LLM {result.expected_fallback_rate:.0%}, {result.latency_ms:.1f} ms/prediction")
pd.DataFrame(result.leaderboard)[["candidate", "metric", "metric_std", "fit_time_s", "predict_latency_ms", "winner"]]

winner tfidf_linear: macro_f1 0.915 (LLM vs human labels: 0.834)
threshold 0.70, expected fallback to the LLM 0%, 0.4 ms/prediction


,candidate,metric,metric_std,fit_time_s,predict_latency_ms,winner
0,tfidf_linear,0.914822,0.010779,0.932216,0.363375,True
1,e5_logreg,0.909300,0.011318,8.835920,0.180230,NaN


## 5. Wrap your LLM function and shadow it
`my_llm` here replays what the "LLM" said; in your code it is whatever you already call.

In [7]:
from pecca.runtime.logging_sink import flush_all

emails = ds.to_pandas().tail(60)
said = {f"{i['subject']}\n\n{i['body']}": out for i, out in zip(emails["input"], emails["llm_output"])}   # the LLM's historical answers

def to_input(text):
    subject, body = text.split("\n\n", 1)
    return {"subject": subject, "body": body}

@pecca.replace("route_rfi", mode="shadow", input_adapter=to_input)
def route_rfi(text: str) -> str:
    return said[text]                     # stand-in for my_llm.classify(text), unchanged

print(pecca.promote("route_rfi", "shadow"))   # governance-gated mode change (default project: metric gates only)
for text in said:
    route_rfi(text)                        # still returns the LLM's answer; Pecca logs the model's next to it
flush_all()
report = pecca.evaluate("route_rfi", "1d")
print(f"{report.n_rows} calls in shadow: agreement with the LLM {report.agreement_rate:.0%}, would fall back {report.fallback_rate:.0%}")

Decision(allowed=True, failing_gates=[], pending_approvals=[], evidence_missing=[], transition='record->shadow', forced=False)
60 calls in shadow: agreement with the LLM 93%, would fall back 0%


## 6. Status

In [8]:
!pecca status default/default/route_rfi

call       default/default/route_rfi
mode       shadow          since 2026-10-06
version    v1              trained 2026-10-06 on 6,000 rows
model      tfidf_linear    macro_f1 0.91  (llm 0.83)
threshold  0.70            expected fallback 0%
agreement  0.93 (14d)      fallback 0%   drift 0.54
policy     shadow_at cv_metric>=0.80  live_at agreement>=0.85 & shadow_days>=7
next       -


## 7. Audit pack preview
Every model version gets lineage, benchmarks, a confusion matrix, threshold rationale, calibration and an sha256 manifest.

In [9]:
from pathlib import Path
from IPython.display import Markdown, display

pack = Path(pecca.audit_pack("route_rfi", out="markdown")).parent
print(sorted(p.name for p in pack.iterdir()))
for section in ("summary", "benchmark_leaderboard", "threshold_rationale"):
    display(Markdown((pack / f"{section}.md").read_text()))

['approvals.md', 'audit.md', 'benchmark_leaderboard.md', 'calibration.md', 'calibration.png', 'confusion_matrix.md', 'confusion_matrix.png', 'controls_mapping.md', 'data_lineage.md', 'drift_report.md', 'environment.md', 'manifest.md', 'manifest.sha256', 'model_card.md', 'per_class_report.md', 'shadow_results.md', 'summary.md', 'threshold_rationale.md']


# Audit pack — default/default/route_rfi v1

Generated 2026-10-06T18:22:53.664312+00:00 · workspace `default` · project `default`

| | |
| --- | --- |
| mode | shadow |
| model | `tfidf_linear` (pickle) |
| macro_f1 (CV) | 0.915 ± 0.011 |
| LLM metric (vs human) | 0.834 |
| threshold / expected fallback | 0.70 / 0.0% |
| trained | 2026-10-06T18:22:50.097368+00:00 |


# Benchmark leaderboard (macro_f1, cross-validated)

| candidate | macro_f1 | std | fit s | latency ms | status |
| --- | --- | --- | --- | --- | --- |
| tfidf_linear ★ | 0.915 | 0.011 | 0.9 | 0.36 | ok |
| e5_logreg | 0.909 | 0.011 | 8.8 | 0.18 | ok |



# Threshold rationale

The threshold is the **smallest** confidence `t` where precision on hold-out rows with `confidence >= t` is at least **0.95**.

| | |
| --- | --- |
| threshold | 0.701 |
| expected fallback rate | 0.0% |
| target met | True |

Rows below the threshold are answered by the original LLM function.



## Next
* Let it run in shadow, then `pecca promote default/default/route_rfi --mode live` once the gates pass.
* Docs: https://pecca-core.github.io/pecca/ · Source: https://github.com/pecca-core/pecca